# [LAB-05] [PBT] 데이터 품질 점검

## 1. Boston Housing - 데이터 품질 점검

### #01 준비작업

1. 필요한 라이브러리 참조

In [162]:
import numpy as np
from jussam import load_data
from pandas import DataFrame, read_excel

2. 데이터 불러오기

In [163]:
origin = load_data('boston')
origin.head()

📚 보스턴 인근 지역의 인구·환경·교육·범죄·세금 정보 등을 기반으로 주택의 중위가격(MEDV)을 예측하기 위해 수집된 대표적인 회귀 분석용 주택시장 데이터 (출처: https://www.kaggle.com/datasets/altavish/boston-housing-dataset)

    field    description
--  -------  ---------------------------------------------------------------------------------------------
 0  CRIM     타운별 1인당 범죄율 (Per capita crime rate by town)
 1  ZN       25,000 sqft 이상 주거용 대지 비율 (Proportion of residential land zoned for large lots)
 2  INDUS    비소매 상업구역 비율 (Proportion of non-retail business acres per town)
 3  CHAS     명목형, 찰스강 인접 여부 (Charles River dummy variable: 1 if tract bounds river; 0 otherwise)
 4  NOX      대기 오염 정도 (Nitric oxides concentration, parts per 10 million)
 5  RM       주택당 평균 방(Room) 수 (Average number of rooms per dwelling)
 6  AGE      1940년 이전 건축 비율 (Proportion of owner-occupied units built prior to 1940)
 7  DIS      5개 주요 고용센터까지의 거리 (Weighted distances to five Boston employment centers)
 8  RAD      고속도로 접근성 지수 (Index of accessibility to radial highways)
 9  T

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,18.000,2.310,0,0.538,6.575,65.200,4.090,1,296,15.300,396.900,4.980,24.000
1,0.027,0.000,7.070,0,0.469,6.421,78.900,4.967,2,242,17.800,396.900,9.140,21.600
2,0.027,0.000,7.070,0,0.469,7.185,61.100,4.967,2,242,17.800,392.830,4.030,34.700
3,0.032,0.000,2.180,0,0.458,6.998,45.800,6.062,3,222,18.700,394.630,2.940,33.400
4,0.069,0.000,2.180,0,0.458,7.147,54.200,6.062,3,222,18.700,396.900,5.330,36.200


### #02 자료형 확인

1. 자료형 확인

In [164]:
origin.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    int64  
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    int64  
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB


2. 자료형 변환

In [165]:
# 원본 데이터 프레임 복사
df1 = origin.copy()
df1["CHAS"] = df1["CHAS"].astype("category")
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   CRIM     506 non-null    float64 
 1   ZN       506 non-null    float64 
 2   INDUS    506 non-null    float64 
 3   CHAS     506 non-null    category
 4   NOX      506 non-null    float64 
 5   RM       506 non-null    float64 
 6   AGE      506 non-null    float64 
 7   DIS      506 non-null    float64 
 8   RAD      506 non-null    int64   
 9   TAX      506 non-null    int64   
 10  PTRATIO  506 non-null    float64 
 11  B        506 non-null    float64 
 12  LSTAT    506 non-null    float64 
 13  MEDV     506 non-null    float64 
dtypes: category(1), float64(11), int64(2)
memory usage: 52.0 KB


### #03 데이터 중복 점검

1. 행 단위 완전 중복 확인

In [166]:
dup = df1.duplicated()
dup

0      False
1      False
2      False
3      False
4      False
       ...  
501    False
502    False
503    False
504    False
505    False
Length: 506, dtype: bool

2. 중복 데이터 제거

In [167]:
df2 = df1.drop_duplicates()
df2.duplicated().sum()

np.int64(0)

### #04 형식, 표기, 단위 점검

1. 명목형 변수의 단위 확인

In [168]:
df2['CHAS'].value_counts()

CHAS
0    471
1     35
Name: count, dtype: int64

2. 연속형 변수의 단위 확인을 위한 변수명 추출

In [169]:
# 값을 검사, 수치형 변수의 이름 추출
fields = df2.select_dtypes(include="number").columns.to_list()
print(fields)

['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']


3. 연속형 변수의 단위 확인

In [170]:
minmax = []     # 최소값과 최대값을 저장할 리스트

for field in fields:
    min_value = df2[field].min()
    max_value = df2[field].max()
    minmax.append({"min": min_value, "max": max_value})

# 최소값과 최대값을 데이터 프레임으로 변환하여 출력
minmax_df = DataFrame(minmax, index=fields)
minmax_df

,min,max
CRIM,0.006,88.976
ZN,0.000,100.000
INDUS,0.460,27.740
NOX,0.385,0.871
RM,3.561,8.780
AGE,2.900,100.000
DIS,1.130,12.127
RAD,1.000,24.000
TAX,187.000,711.000
PTRATIO,12.600,22.000


### #05 결측치 점검

1. 결측치 확인

In [171]:
na_count = df2.isna().sum()
na_count

CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64

2. 결측치 비율 계산을 위한 데이터 크기 확인

In [172]:
# 데이터의 행과 열의 수 확인
rows, cols = df2.shape
print(f"rows: {rows}, cols: {cols}")

rows: 506, cols: 14


3. 결측치 비율 확인

In [173]:
na_ratio = na_count / rows
na_ratio

CRIM      0.000
ZN        0.000
INDUS     0.000
CHAS      0.000
NOX       0.000
RM        0.000
AGE       0.000
DIS       0.000
RAD       0.000
TAX       0.000
PTRATIO   0.000
B         0.000
LSTAT     0.000
MEDV      0.000
dtype: float64

### #06 품질 점검 결과 저장

1. 엑셀 파일로 저장하기

In [174]:
df2.to_excel("boston_qtcheck.xlsx", index=False)

## 2. Boston Housing - 기술 통계량 확인

### #01 준비작업

1. 데이터 불러오기

In [175]:
# origin_qt = read_excel("boston_qtcheck.xlsx")

# 라이브러리를 통해서도 로드 가능함
origin_qt = load_data("boston_qtcheck")

📚 보스턴 주택 가격 데이터셋의 품질 검사 완료 버전 (출처: 자체 정제)

    field    description
--  -------  ---------------------------------------------------------------------------------------------
 0  CRIM     타운별 1인당 범죄율 (Per capita crime rate by town)
 1  ZN       25,000 sqft 이상 주거용 대지 비율 (Proportion of residential land zoned for large lots)
 2  INDUS    비소매 상업구역 비율 (Proportion of non-retail business acres per town)
 3  CHAS     명목형, 찰스강 인접 여부 (Charles River dummy variable: 1 if tract bounds river; 0 otherwise)
 4  NOX      대기 오염 정도 (Nitric oxides concentration, parts per 10 million)
 5  RM       주택당 평균 방(Room) 수 (Average number of rooms per dwelling)
 6  AGE      1940년 이전 건축 비율 (Proportion of owner-occupied units built prior to 1940)
 7  DIS      5개 주요 고용센터까지의 거리 (Weighted distances to five Boston employment centers)
 8  RAD      고속도로 접근성 지수 (Index of accessibility to radial highways)
 9  TAX      재산세율 (Full-value property-tax rate per $10,000)
10  PTRATIO  교육구 학생-교사 비율 (Pupil–teacher ratio by town)
11  

2. 타입 변환

In [176]:
df3 = origin_qt.copy()
df3['CHAS'] = df3['CHAS'].astype('category')
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   CRIM     506 non-null    float64 
 1   ZN       506 non-null    float64 
 2   INDUS    506 non-null    float64 
 3   CHAS     506 non-null    category
 4   NOX      506 non-null    float64 
 5   RM       506 non-null    float64 
 6   AGE      506 non-null    float64 
 7   DIS      506 non-null    float64 
 8   RAD      506 non-null    int64   
 9   TAX      506 non-null    int64   
 10  PTRATIO  506 non-null    float64 
 11  B        506 non-null    float64 
 12  LSTAT    506 non-null    float64 
 13  MEDV     506 non-null    float64 
dtypes: category(1), float64(11), int64(2)
memory usage: 52.0 KB


### #02 기술 통계량 표 생성

1. 연속형 변수의 기술 통계량 표

In [177]:
# T는 데이터프레임의 행과 열을 바꿔주는 역할
desc_df = df3.describe().T
desc_df

,count,mean,std,min,25%,50%,75%,max
CRIM,506.000,3.614,8.602,0.006,0.082,0.257,3.677,88.976
ZN,506.000,11.364,23.322,0.000,0.000,0.000,12.500,100.000
INDUS,506.000,11.137,6.860,0.460,5.190,9.690,18.100,27.740
NOX,506.000,0.555,0.116,0.385,0.449,0.538,0.624,0.871
RM,506.000,6.285,0.703,3.561,5.885,6.208,6.623,8.780
AGE,506.000,68.575,28.149,2.900,45.025,77.500,94.075,100.000
DIS,506.000,3.795,2.106,1.130,2.100,3.207,5.188,12.127
RAD,506.000,9.549,8.707,1.000,4.000,5.000,24.000,24.000
TAX,506.000,408.237,168.537,187.000,279.000,330.000,666.000,711.000
PTRATIO,506.000,18.456,2.165,12.600,17.400,19.050,20.200,22.000


2. 명목형 변수의 기술 통계량 표

In [178]:
cate_desc_df = df3.describe(include='category').T
cate_desc_df

,count,unique,top,freq
CHAS,506,2,0,471


3. 명목형 변수의 기술 통계량

In [179]:
# 명목형 변수의 필드명 추출
cate_fields = df3.select_dtypes(include='category').columns

# 명목형 변수의 고유값과 빈도 계산
for field in cate_fields:
    vcount = df3[field].value_counts()      # 값의 종류별 개수
    percent = vcount / df3.shape[0]         # 전체 데이터 길이 대비 비율

    df = DataFrame({'count': vcount, 'percent': percent})       # 결과 저장
    display(df)         # 결과 출력

,count,percent
CHAS,,
0,471,0.931
1,35,0.069


### #03 중심 수준 파악

1. 평균-중앙값의 상대 차이율 계산

In [180]:
# "평균-중앙값 상대 차이율 = |평균 - 중앙값| / 중앙값" 컬럼 추가
desc_df['rel_diff'] = abs(desc_df['mean'] - desc_df['50%']) / desc_df['50%']

# 상대 차이율 의미 컬럼 추가
conditions = [desc_df['rel_diff'] < 0.1, desc_df['rel_diff'] < 0.5]
choices = ['similar', 'diff']
desc_df['rdiff_flag'] = np.select(conditions, choices, default='large_diff')

# 필요한 컬럼만 선택하여 출력
desc_df[['mean', '50%', 'rel_diff', 'rdiff_flag']]

,mean,50%,rel_diff,rdiff_flag
CRIM,3.614,0.257,13.087,large_diff
ZN,11.364,0.000,inf,large_diff
INDUS,11.137,9.690,0.149,diff
NOX,0.555,0.538,0.031,similar
RM,6.285,6.208,0.012,similar
AGE,68.575,77.500,0.115,diff
DIS,3.795,3.207,0.183,diff
RAD,9.549,5.000,0.910,large_diff
TAX,408.237,330.000,0.237,diff
PTRATIO,18.456,19.050,0.031,similar


### #04 이상치 신호 감지

1. IQR, 이상치 경계값 계산

In [181]:
# iqr
desc_df['iqr'] = desc_df['75%'] - desc_df['25%']

# 상한 이상치 경계
desc_df['upper_bound'] = desc_df['75%'] + 1.5 * desc_df['iqr']

# 하한 이상치 경계
desc_df['lower_bound'] = desc_df['25%'] - 1.5 * desc_df['iqr']

# 필요한 컬럼만 선택하여 출력
desc_df[['iqr', 'upper_bound', 'lower_bound']]

,iqr,upper_bound,lower_bound
CRIM,3.595,9.070,-5.311
ZN,12.500,31.250,-18.750
INDUS,12.910,37.465,-14.175
NOX,0.175,0.886,0.187
RM,0.738,7.731,4.778
AGE,49.050,167.650,-28.550
DIS,3.088,9.821,-2.532
RAD,20.000,54.000,-26.000
TAX,387.000,1246.500,-301.500
PTRATIO,2.800,24.400,13.200


2. 명목형 변수를 제외한 데이터 프레임

In [182]:
# 명목형 변수의 필드명 추출
cate_fields = df3.select_dtypes(include='category').columns

df4 = df3.drop(columns=cate_fields)
df4.head()

,CRIM,ZN,INDUS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,18.000,2.310,0.538,6.575,65.200,4.090,1,296,15.300,396.900,4.980,24.000
1,0.027,0.000,7.070,0.469,6.421,78.900,4.967,2,242,17.800,396.900,9.140,21.600
2,0.027,0.000,7.070,0.469,7.185,61.100,4.967,2,242,17.800,392.830,4.030,34.700
3,0.032,0.000,2.180,0.458,6.998,45.800,6.062,3,222,18.700,394.630,2.940,33.400
4,0.069,0.000,2.180,0.458,7.147,54.200,6.062,3,222,18.700,396.900,5.330,36.200


3. 상한 이상치 탐지

In [183]:
# 상한 이상치 탐지
desc_df['upper_outliers'] = ((df4 > desc_df['upper_bound'])).sum()

# 상한 이상치 수 비율
desc_df['upper_outliers_ratio'] = desc_df['upper_outliers'] / df4.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['upper_outliers', 'upper_outliers_ratio']]

,upper_outliers,upper_outliers_ratio
CRIM,66,0.130
ZN,68,0.134
INDUS,0,0.000
NOX,0,0.000
RM,22,0.043
AGE,0,0.000
DIS,5,0.010
RAD,0,0.000
TAX,0,0.000
PTRATIO,0,0.000


4. 하한 이상치 탐지

In [184]:
# 상한 이상치 탐지
desc_df['lower_outliers'] = ((df4 < desc_df['lower_bound'])).sum()

# 상한 이상치 수 비율
desc_df['lower_outliers_ratio'] = desc_df['lower_outliers'] / df4.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['lower_outliers', 'lower_outliers_ratio']]

,lower_outliers,lower_outliers_ratio
CRIM,0,0.000
ZN,0,0.000
INDUS,0,0.000
NOX,0,0.000
RM,8,0.016
AGE,0,0.000
DIS,0,0.000
RAD,0,0.000
TAX,0,0.000
PTRATIO,15,0.030


5. 전체 이상치 집계

In [185]:
# 통합 이상치 수
desc_df['outliers'] = desc_df['upper_outliers'] + desc_df['lower_outliers']

# 통합 이상치 수 비율
desc_df['outliers_ratio'] = desc_df['outliers'] / df4.shape[0]

# 이상치 탐지 결과 확인
desc_df[['upper_outliers', 'upper_outliers_ratio', 'lower_outliers', 'lower_outliers_ratio', 'outliers', 'outliers_ratio']]

,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio
CRIM,66,0.130,0,0.000,66,0.130
ZN,68,0.134,0,0.000,68,0.134
INDUS,0,0.000,0,0.000,0,0.000
NOX,0,0.000,0,0.000,0,0.000
RM,22,0.043,8,0.016,30,0.059
AGE,0,0.000,0,0.000,0,0.000
DIS,5,0.010,0,0.000,5,0.010
RAD,0,0.000,0,0.000,0,0.000
TAX,0,0.000,0,0.000,0,0.000
PTRATIO,0,0.000,15,0.030,15,0.030


### #05 비대칭 신호 확인

1. 왜도 점검

In [186]:
# 왜도 계산
desc_df['skew'] = df4.skew()

# 왜도를 통한 분포 형태 해석
conditions_skew = [(desc_df['skew'] < -0.5), (desc_df['skew'] > 0.5)]
choices_skew = ['left tail', 'right tail']
desc_df['skew_interpret'] = np.select(conditions_skew, choices_skew, default='symmetric')

# 필요한 컬럼만 선택하여 추출
desc_df[['skew', 'skew_interpret']]

,skew,skew_interpret
CRIM,5.223,right tail
ZN,2.226,right tail
INDUS,0.295,symmetric
NOX,0.729,right tail
RM,0.404,symmetric
AGE,-0.599,left tail
DIS,1.012,right tail
RAD,1.005,right tail
TAX,0.670,right tail
PTRATIO,-0.802,left tail


2. 첨도 점검

In [187]:
# 첨도 계산
desc_df['kurt'] = df4.kurt()

# 첨도를 통한 분포 해석
conditions_kurt = [(desc_df['kurt'] < 0), (desc_df['kurt'] > 0)]
choices_kurt = ['platykurtic', 'leptokurtic']
desc_df['kurt_interpret'] = np.select(conditions_kurt, choices_kurt, default='mesokurtic')

# 필요한 컬럼만 선택하여 추출
desc_df[['kurt', 'kurt_interpret']]

,kurt,kurt_interpret
CRIM,37.131,leptokurtic
ZN,4.032,leptokurtic
INDUS,-1.234,platykurtic
NOX,-0.065,platykurtic
RM,1.892,leptokurtic
AGE,-0.968,platykurtic
DIS,0.488,leptokurtic
RAD,-0.867,platykurtic
TAX,-1.142,platykurtic
PTRATIO,-0.285,platykurtic


3. 로그 변환 필요성 판단 함수 정의

In [188]:
def judge_log_transform(skew, kurt, min_value):
    right = skew > 1 or (skew > 0.5 and kurt > 0)       # 우측 꼬리
    left = skew <= -1 or (skew < -0.5 and kurt > 0)     # 좌측 꼬리

    if right:
        if min_value > 0: return "log"      # 0이 없으면 순수 log가 정의된다
        if min_value == 0: return "log1p"   # 0을 피하려고 +1한다
        return "none"                       # 음수가 섞이면 로그 계열을 쓸 수 없다
    if left:
        return "reverse_log1p"              # max-x = 0 이라 항상 안전하다
    return "none"

4. 로그 변환 필요성 판정

In [189]:
desc_df['log_need'] = desc_df.apply(lambda row: judge_log_transform(row['skew'], row['kurt'], row['min']), axis=1)

# 로그 변환 필요성 결과 확인
desc_df[['skew', 'kurt', 'log_need']]

,skew,kurt,log_need
CRIM,5.223,37.131,log
ZN,2.226,4.032,log1p
INDUS,0.295,-1.234,none
NOX,0.729,-0.065,none
RM,0.404,1.892,none
AGE,-0.599,-0.968,none
DIS,1.012,0.488,log
RAD,1.005,-0.867,log
TAX,0.670,-1.142,none
PTRATIO,-0.802,-0.285,none


### #06 최종 기술 통계량 확인

1. 최종 기술 통계량 확인

In [190]:
desc_df.T

,CRIM,ZN,INDUS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
count,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000,506.000
mean,3.614,11.364,11.137,0.555,6.285,68.575,3.795,9.549,408.237,18.456,356.674,12.653,22.533
std,8.602,23.322,6.860,0.116,0.703,28.149,2.106,8.707,168.537,2.165,91.295,7.141,9.197
min,0.006,0.000,0.460,0.385,3.561,2.900,1.130,1.000,187.000,12.600,0.320,1.730,5.000
25%,0.082,0.000,5.190,0.449,5.885,45.025,2.100,4.000,279.000,17.400,375.377,6.950,17.025
50%,0.257,0.000,9.690,0.538,6.208,77.500,3.207,5.000,330.000,19.050,391.440,11.360,21.200
75%,3.677,12.500,18.100,0.624,6.623,94.075,5.188,24.000,666.000,20.200,396.225,16.955,25.000
max,88.976,100.000,27.740,0.871,8.780,100.000,12.127,24.000,711.000,22.000,396.900,37.970,50.000
rel_diff,13.087,inf,0.149,0.031,0.012,0.115,0.183,0.910,0.237,0.031,0.089,0.114,0.063
rdiff_flag,large_diff,large_diff,diff,similar,similar,diff,diff,large_diff,diff,similar,similar,diff,similar


2. 기술 통계량 표 저장

In [191]:
desc_df.to_excel("boston_qtcheck_desc.xlsx")

## 연습문제

### 보험료 가격 결정 구조 분석을 위한 데이터셋의 품질 검사

In [192]:
# 필요한 라이브러리 참조
import numpy as np
from jussam import load_data
from pandas import DataFrame, read_excel, get_dummies

In [193]:
# 데이터 불러오기
df = load_data('insurance')
df.head()

📚 개인의 나이·성별·BMI·흡연 여부·거주 지역 등 기본 건강·인구학적 정보를 바탕으로 의료보험 청구 비용(charges)을 예측하도록 구성된, 선형회귀와 머신러닝 실습에 널리 사용되는 대표적인 보험 비용 데이터셋 (출처: https://www.kaggle.com/datasets/mirichoi0218/insurance)

    field     description
--  --------  --------------------------------------------------------------
 0  age       가입자의 나이(세)
 1  sex       성별 (male, female)
 2  bmi       체질량 지수(Body Mass Index)
 3  children  부양 자녀 수(보험 내 자녀 수)
 4  smoker    흡연 여부 (yes / no)
 5  region    미국 내 거주 지역 (northeast, northwest, southeast, southwest)
 6  charges   의료보험 청구 비용(달러). 예측해야 하는 타깃 변수.



,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


In [194]:
# 데이터 구조 살펴보기
df.shape

(1338, 7)

In [195]:
# 자료형 확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB


In [196]:
# 타입 변환 str > category
df1 = df.copy()
df1["sex"] = df1["sex"].astype("category")
df1["smoker"] = df1["smoker"].astype("category")
df1["region"] = df1["region"].astype("category")

df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1338 non-null   int64   
 1   sex       1338 non-null   category
 2   bmi       1338 non-null   float64 
 3   children  1338 non-null   int64   
 4   smoker    1338 non-null   category
 5   region    1338 non-null   category
 6   charges   1338 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 46.0 KB


In [197]:
# 데이터 중복 확인
dup = df1.duplicated()
dup.sum()

np.int64(1)

> 중복 데이터가 발견되었다.

In [198]:
# 데이터 중복 제거
df2 = df1.drop_duplicates()
df2.duplicated().sum()

np.int64(0)

In [199]:
# 명목형 변수에 대한 값 종류 확인
# sex_count = df2['sex'].value_counts()
# smoker_count = df2['smoker'].value_counts()
# region_count = df2['region'].value_counts()

# display(sex_count, smoker_count, region_count)

fields = df2.select_dtypes(include=["category"]).columns

for field in fields:
    display(df2[field].value_counts())

sex
male      675
female    662
Name: count, dtype: int64

smoker
no     1063
yes     274
Name: count, dtype: int64

region
southeast    364
southwest    325
northeast    324
northwest    324
Name: count, dtype: int64

> 데이터의 종류에는 문제가 없으나, 라벨링 처리는 필요하다.

In [200]:
# 데이터 라벨링 - 더미변수 만들기
# df3 = get_dummies(df2, columns=['sex', 'smoker', 'region'], dtype='int')
# df3.head()

In [201]:
# 연속형 변수의 변수명 추출
fields = df2.select_dtypes(include="number").columns.to_list()
fields

['age', 'bmi', 'children', 'charges']

In [202]:
# 연속형 변수의 단위 확인
minmax = []     # 최소값과 최대값을 저장할 리스트

for field in fields:
    min_value = df2[field].min()
    max_value = df2[field].max()
    minmax.append({"min": min_value, "max": max_value})

# 최소값과 최대값을 데이터 프레임으로 변환하여 출력
minmax_df = DataFrame(minmax, index=fields)
minmax_df

,min,max
age,18.000,64.000
bmi,15.960,53.130
children,0.000,5.000
charges,1121.874,63770.428


> 데이터 명세의 내용에서 크게 벗어나는 범위는 없는 것으로 판단됨

In [203]:
# 결측치 확인
na_count = df2.isna().sum()
na_count

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [204]:
# 결측치 비율 확인
na_ratio = na_count / rows
na_ratio

age        0.000
sex        0.000
bmi        0.000
children   0.000
smoker     0.000
region     0.000
charges    0.000
dtype: float64

In [205]:
# 품질 점검 결과 저장
df2.to_excel("insurance_qtcheck.xlsx", index=False)

In [206]:
# category 타입으로의 변환은 데이터를 불러올 때 마다 진행해야 한다.
df3 = df2.copy()
df3["sex"] = df3["sex"].astype("category")
df3["smoker"] = df3["smoker"].astype("category")
df3["region"] = df3["region"].astype("category")
df3.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1337 non-null   int64   
 1   sex       1337 non-null   category
 2   bmi       1337 non-null   float64 
 3   children  1337 non-null   int64   
 4   smoker    1337 non-null   category
 5   region    1337 non-null   category
 6   charges   1337 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 56.6 KB


In [207]:
desc_df = df3.describe().T
desc_df

,count,mean,std,min,25%,50%,75%,max
age,1337.000,39.222,14.044,18.000,27.000,39.000,51.000,64.000
bmi,1337.000,30.663,6.100,15.960,26.290,30.400,34.700,53.130
children,1337.000,1.096,1.206,0.000,0.000,1.000,2.000,5.000
charges,1337.000,13279.121,12110.360,1121.874,4746.344,9386.161,16657.717,63770.428


In [208]:
# 명목형 변수의 기술 통계량 표
cate_desc_df = df3.describe(include='category').T
cate_desc_df

,count,unique,top,freq
sex,1337,2,male,675
smoker,1337,2,no,1063
region,1337,4,southeast,364


In [209]:
# 명목형 변수의 기술 통계량
cate_fields = df3.select_dtypes(include='category').columns

# 명목형 변수의 고유값과 빈도 계산
for field in cate_fields:
    vcount = df3[field].value_counts()      # 값의 종류별 개수
    percent = vcount / df3.shape[0]         # 전체 데이터 길이 대비 비율

    df = DataFrame({'count': vcount, 'percent': percent})       # 결과 저장
    display(df)         # 결과 출력

,count,percent
sex,,
male,675,0.505
female,662,0.495


,count,percent
smoker,,
no,1063,0.795
yes,274,0.205


,count,percent
region,,
southeast,364,0.272
southwest,325,0.243
northeast,324,0.242
northwest,324,0.242


In [210]:
# 중심 수준 파악
# "평균-중앙값 상대 차이율 = |평균 - 중앙값| / 중앙값" 컬럼 추가
desc_df['rel_diff'] = abs(desc_df['mean'] - desc_df['50%']) / desc_df['50%']

# 상대 차이율 의미 컬럼 추가
conditions = [desc_df['rel_diff'] < 0.1, desc_df['rel_diff'] < 0.5]
choices = ['similar', 'diff']
desc_df['rdiff_flag'] = np.select(conditions, choices, default='large_diff')

# 필요한 컬럼만 선택하여 출력
desc_df[['mean', '50%', 'rel_diff', 'rdiff_flag']]

,mean,50%,rel_diff,rdiff_flag
age,39.222,39.000,0.006,similar
bmi,30.663,30.400,0.009,similar
children,1.096,1.000,0.096,similar
charges,13279.121,9386.161,0.415,diff


In [211]:
# 이상치 신호 감지

# iqr
desc_df['iqr'] = desc_df['75%'] - desc_df['25%']

# 상한 이상치 경계
desc_df['upper_bound'] = desc_df['75%'] + 1.5 * desc_df['iqr']

# 하한 이상치 경계
desc_df['lower_bound'] = desc_df['25%'] - 1.5 * desc_df['iqr']

# 필요한 컬럼만 선택하여 출력
desc_df[['iqr', 'upper_bound', 'lower_bound']]

,iqr,upper_bound,lower_bound
age,24.000,87.000,-9.000
bmi,8.410,47.315,13.675
children,2.000,5.000,-3.000
charges,11911.373,34524.778,-13120.716


In [223]:
# 명목형 변수의 필드명 추출
cate_fields = df3.select_dtypes(include='category').columns

df4 = df3.drop(columns=cate_fields)
df4.head()

,age,bmi,children,charges
0,19,27.900,0,16884.924
1,18,33.770,1,1725.552
2,28,33.000,3,4449.462
3,33,22.705,0,21984.471
4,32,28.880,0,3866.855


In [224]:
# 상한 이상치 탐지
desc_df['upper_outliers'] = ((df4 > desc_df['upper_bound'])).sum()

# 상한 이상치 수 비율
desc_df['upper_outliers_ratio'] = desc_df['upper_outliers'] / df3.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['upper_outliers', 'upper_outliers_ratio']]

,upper_outliers,upper_outliers_ratio
age,0,0.000
bmi,9,0.007
children,0,0.000
charges,139,0.104


In [225]:
# 상한 이상치 탐지
desc_df['lower_outliers'] = ((df4 < desc_df['lower_bound'])).sum()

# 상한 이상치 수 비율
desc_df['lower_outliers_ratio'] = desc_df['lower_outliers'] / df4.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['lower_outliers', 'lower_outliers_ratio']]

,lower_outliers,lower_outliers_ratio
age,0,0.000
bmi,0,0.000
children,0,0.000
charges,0,0.000


In [226]:
# 통합 이상치 수
desc_df['outliers'] = desc_df['upper_outliers'] + desc_df['lower_outliers']

# 통합 이상치 수 비율
desc_df['outliers_ratio'] = desc_df['outliers'] / df4.shape[0]

# 이상치 탐지 결과 확인
desc_df[['upper_outliers', 'upper_outliers_ratio', 'lower_outliers', 'lower_outliers_ratio', 'outliers', 'outliers_ratio']]

,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio
age,0,0.000,0,0.000,0,0.000
bmi,9,0.007,0,0.000,9,0.007
children,0,0.000,0,0.000,0,0.000
charges,139,0.104,0,0.000,139,0.104


In [216]:
# 왜도 계산
desc_df['skew'] = df4.skew()

# 왜도를 통한 분포 형태 해석
conditions_skew = [(desc_df['skew'] < -0.5), (desc_df['skew'] > 0.5)]
choices_skew = ['left tail', 'right tail']
desc_df['skew_interpret'] = np.select(conditions_skew, choices_skew, default='symmetric')

# 필요한 컬럼만 선택하여 추출
desc_df[['skew', 'skew_interpret']]

,skew,skew_interpret
age,0.055,symmetric
bmi,0.284,symmetric
children,0.937,right tail
charges,1.515,right tail


In [217]:
# 첨도 계산
desc_df['kurt'] = df4.kurt()

# 첨도를 통한 분포 해석
conditions_kurt = [(desc_df['kurt'] < 0), (desc_df['kurt'] > 0)]
choices_kurt = ['platykurtic', 'leptokurtic']
desc_df['kurt_interpret'] = np.select(conditions_kurt, choices_kurt, default='mesokurtic')

# 필요한 컬럼만 선택하여 추출
desc_df[['kurt', 'kurt_interpret']]

,kurt,kurt_interpret
age,-1.244,platykurtic
bmi,-0.053,platykurtic
children,0.201,leptokurtic
charges,1.604,leptokurtic


In [218]:
def judge_log_transform(skew, kurt, min_value):
    right = skew > 1 or (skew > 0.5 and kurt > 0)       # 우측 꼬리
    left = skew <= -1 or (skew < -0.5 and kurt > 0)     # 좌측 꼬리

    if right:
        if min_value > 0: return "log"      # 0이 없으면 순수 log가 정의된다
        if min_value == 0: return "log1p"   # 0을 피하려고 +1한다
        return "none"                       # 음수가 섞이면 로그 계열을 쓸 수 없다
    if left:
        return "reverse_log1p"              # max-x = 0 이라 항상 안전하다
    return "none"

In [219]:
desc_df['log_need'] = desc_df.apply(lambda row: judge_log_transform(row['skew'], row['kurt'], row['min']), axis=1)

# 로그 변환 필요성 결과 확인
desc_df[['skew', 'kurt', 'log_need']]

,skew,kurt,log_need
age,0.055,-1.244,none
bmi,0.284,-0.053,none
children,0.937,0.201,log1p
charges,1.515,1.604,log


In [220]:
desc_df.T

,age,bmi,children,charges
count,1337.000,1337.000,1337.000,1337.000
mean,39.222,30.663,1.096,13279.121
std,14.044,6.100,1.206,12110.360
min,18.000,15.960,0.000,1121.874
25%,27.000,26.290,0.000,4746.344
50%,39.000,30.400,1.000,9386.161
75%,51.000,34.700,2.000,16657.717
max,64.000,53.130,5.000,63770.428
rel_diff,0.006,0.009,0.096,0.415
rdiff_flag,similar,similar,similar,diff


In [221]:
# 기술 통계량 표 저장
desc_df.to_excel("insurance_qtcheck_desc.xlsx")

## 💡 인사이트

**1. Charges (의료보험 청구 비용) - 주의 필요**
- 평균(13,279)과 중앙값(9,386) 차이가 큼 (상대차이율 0.415) → 비대칭 분포
- 왜도 1.515로 **강한 우측 꼬리 분포** → 고액 의료비가 우측에 집중
- 상한 이상치 139개(10.4%) → 상당한 극값 존재
- **로그 변환(log1p) 필수** 권장

**2. Children (부양 자녀 수)**
- 왜도 0.937로 우측 꼬리 분포
- 이상치는 없으나 **로그 변환 권장** (분포 정상화)

**3. Age, BMI - 건강함**
- 평균-중앙값 거의 일치 (rel_diff < 0.01)
- 대칭적 분포, 이상치 거의 없음
- 추가 처리 불필요

**4. 데이터 품질**
- 결측치 0건, 완전중복 1건 제거 완료 (1,338→1,337)
- 전체적으로 양호한 상태